# Dynamic Light Scattering, Zeta Potential, and Growth Inhibition Measurements for Self-Associating Amphiphiles Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.tnk3-xhn0/fair2.json`

### Notes
- All dataset entities should be referenced by their `@id` values whenever available.
- This notebook uses variables for record set identifiers and field identifiers so the workflow is easy to adapt.
- The supplied FAIR^2 package summary indicates that the top-level dataset metadata is present, but the `recordSet` list may be empty or may need to be discovered dynamically from the Croissant document at runtime.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`, then inspect high-level dataset information.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.tnk3-xhn0/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata

print("Dataset name:", getattr(metadata, "name", None))
print("Description:", getattr(metadata, "description", None))
print("Identifier:", getattr(metadata, "identifier", None))
print("Version:", getattr(metadata, "version", None))
print("License:", getattr(metadata, "license", None))

In [ ]:
# Convert metadata to a JSON-like structure for optional deeper inspection
metadata_json = metadata.to_json()
print(type(metadata_json))
print(json.dumps({
    "name": metadata_json.get("name"),
    "description": metadata_json.get("description"),
    "identifier": metadata_json.get("identifier"),
    "version": metadata_json.get("version"),
    "datePublished": metadata_json.get("datePublished")
}, indent=2))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Because Croissant datasets can expose their structure in slightly different ways, the next cells first inspect the metadata JSON and then build a compact overview table.

In [ ]:
# Inspect top-level keys in the metadata JSON
sorted(metadata_json.keys())

In [ ]:
# Extract record set definitions, if present
record_sets_meta = metadata_json.get("recordSet", []) or []
print(f"Number of record sets declared in metadata: {len(record_sets_meta)}")
record_sets_meta[:2] if len(record_sets_meta) > 0 else "No record sets listed in top-level metadata JSON."

In [ ]:
# Build a structured overview of record sets and fields using @id references
record_set_rows = []
field_rows = []

for rs in record_sets_meta:
    rs_id = rs.get("@id")
    rs_name = rs.get("name")
    rs_desc = rs.get("description")
    record_set_rows.append({
        "record_set_id": rs_id,
        "record_set_name": rs_name,
        "record_set_description": rs_desc,
        "field_count": len(rs.get("field", []) or [])
    })
    
    for fld in rs.get("field", []) or []:
        field_rows.append({
            "record_set_id": rs_id,
            "field_id": fld.get("@id"),
            "field_name": fld.get("name"),
            "field_description": fld.get("description"),
            "data_type": fld.get("dataType"),
            "source": fld.get("source"),
            "references_column": fld.get("column")
        })

record_sets_df = pd.DataFrame(record_set_rows)
fields_df = pd.DataFrame(field_rows)

display(record_sets_df)
display(fields_df.head(20))

The next cell prints sample records from the first available record set using its `@id`. If no record sets are present in the metadata, it will fail gracefully and explain why.

In [ ]:
# Print a few sample records from the first available record set
if not record_sets_df.empty:
    first_record_set_id = record_sets_df.loc[0, "record_set_id"]
    print("Using record set @id:", first_record_set_id)
    
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets were found in the dataset metadata. This dataset may expose metadata only, or record sets may need to be resolved externally.")

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame for analysis. All record set references use `@id` values.

In [ ]:
# Extract records from each record set into DataFrames
record_set_ids = record_sets_df["record_set_id"].dropna().tolist() if not record_sets_df.empty else []
dataframes = {}
load_summary = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        load_summary.append({
            "record_set_id": record_set_id,
            "n_rows": len(df),
            "n_columns": len(df.columns),
            "columns": list(df.columns)
        })
    except Exception as e:
        load_summary.append({
            "record_set_id": record_set_id,
            "n_rows": None,
            "n_columns": None,
            "columns": [],
            "error": str(e)
        })

load_summary_df = pd.DataFrame(load_summary)
display(load_summary_df)

In [ ]:
# Preview the first successfully loaded DataFrame
loaded_record_set_ids = [rs_id for rs_id, df in dataframes.items() if isinstance(df, pd.DataFrame)]

if loaded_record_set_ids:
    preview_record_set_id = loaded_record_set_ids[0]
    print("Preview record set @id:", preview_record_set_id)
    print("Columns:")
    print(dataframes[preview_record_set_id].columns.tolist())
    display(dataframes[preview_record_set_id].head())
else:
    print("No record sets were successfully loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply reusable data-processing steps:
- identify numeric and categorical fields
- filter records based on a numeric threshold
- normalize a selected numeric field
- group by a categorical field when available
- summarize missing values

This section is written defensively so it can adapt to whichever record sets and field IDs are actually available.

In [ ]:
# Choose a DataFrame for EDA
if loaded_record_set_ids:
    eda_record_set_id = loaded_record_set_ids[0]
    eda_df = dataframes[eda_record_set_id].copy()
    print("EDA record set @id:", eda_record_set_id)
    print("Shape:", eda_df.shape)
else:
    eda_record_set_id = None
    eda_df = pd.DataFrame()
    print("No DataFrame available for EDA.")

In [ ]:
# Detect numeric and categorical columns dynamically
if not eda_df.empty:
    inferred_numeric_columns = eda_df.select_dtypes(include="number").columns.tolist()
    inferred_non_numeric_columns = eda_df.select_dtypes(exclude="number").columns.tolist()
    
    print("Numeric columns:", inferred_numeric_columns)
    print("Non-numeric columns:", inferred_non_numeric_columns[:20])
else:
    inferred_numeric_columns = []
    inferred_non_numeric_columns = []

In [ ]:
# Basic missing-value summary
if not eda_df.empty:
    missing_summary = (
        eda_df.isna()
        .sum()
        .sort_values(ascending=False)
        .rename("missing_count")
        .reset_index()
        .rename(columns={"index": "field_id"})
    )
    display(missing_summary.head(20))
else:
    print("Missing-value summary unavailable because no EDA DataFrame was loaded.")

In [ ]:
# Filter and normalize a numeric field dynamically
if not eda_df.empty and inferred_numeric_columns:
    numeric_field_id = inferred_numeric_columns[0]
    numeric_series = eda_df[numeric_field_id].dropna()
    
    if not numeric_series.empty:
        threshold = numeric_series.quantile(0.5)
        filtered_df = eda_df[eda_df[numeric_field_id] > threshold].copy()
        print(f"Selected numeric field @id: {numeric_field_id}")
        print(f"Threshold (median): {threshold}")
        print(f"Filtered rows: {len(filtered_df)}")
        
        std_value = filtered_df[numeric_field_id].std()
        if std_value is not None and pd.notna(std_value) and std_value != 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / std_value
            )
        else:
            filtered_df[f"{numeric_field_id}_normalized"] = 0.0
        
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        numeric_field_id = None
        filtered_df = pd.DataFrame()
        print("The selected numeric field contains no non-null values.")
else:
    numeric_field_id = None
    filtered_df = pd.DataFrame()
    print("No numeric fields available for filtering and normalization.")

In [ ]:
# Group by a categorical field if one is available
if not filtered_df.empty and inferred_non_numeric_columns:
    group_field_id = inferred_non_numeric_columns[0]
    print(f"Selected grouping field @id: {group_field_id}")
    
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id]
        .agg(["count", "mean", "median", "min", "max"])
        .sort_values("count", ascending=False)
    )
    display(grouped_df.head(10))
else:
    group_field_id = None
    grouped_df = pd.DataFrame()
    print("No suitable categorical field was available for grouping.")

## 5. Visualization
Visualize the selected numeric field and, when possible, its relationship to a categorical grouping field.

In [ ]:
# Histogram of the selected numeric field
if not eda_df.empty and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(eda_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("Histogram not generated because no numeric field was available.")

In [ ]:
# Boxplot by grouping field when available
if not eda_df.empty and numeric_field_id is not None and group_field_id is not None:
    top_groups = eda_df[group_field_id].astype(str).value_counts().head(10).index.tolist()
    plot_df = eda_df[eda_df[group_field_id].astype(str).isin(top_groups)].copy()
    
    if not plot_df.empty:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=plot_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Boxplot not generated because filtered plotting data was empty.")
else:
    print("Boxplot not generated because a numeric or grouping field was unavailable.")

In [ ]:
# Correlation heatmap for numeric columns when at least two exist
if not eda_df.empty and len(inferred_numeric_columns) >= 2:
    corr_df = eda_df[inferred_numeric_columns].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_df, cmap="coolwarm", center=0)
    plt.title("Correlation heatmap of numeric fields")
    plt.tight_layout()
    plt.show()
else:
    print("Correlation heatmap not generated because fewer than two numeric fields were available.")

## 6. Conclusion
This notebook demonstrated how to:
- load a Croissant dataset with `mlcroissant`
- inspect dataset metadata without treating `dataset.metadata` as a list or dictionary object directly
- discover record sets and fields using their `@id` values
- extract records into pandas DataFrames
- perform basic EDA, including filtering, normalization, grouping, and missing-value analysis
- create simple visualizations for numeric and grouped data

For this FAIR^2 dataset, the exact available record sets depend on what the Croissant document exposes at runtime. If the metadata contains no `recordSet` definitions, the notebook still provides a reusable exploration pattern and graceful checks for metadata-only cases.